In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# 1. Load the cleaned data
df = pd.read_csv('../data/processed/cleaned_apl_logistics.csv')

# 2. Select features (AVOIDING Data Leakage)
features = ['shipping_mode', 'customer_segment', 'market', 'order_region', 'category_name']
target = 'late_delivery_risk'

ml_df = df[features + [target]].copy()

# 3. Encode categorical features into numbers
label_encoders = {}
for col in features:
    le = LabelEncoder()
    ml_df[col] = le.fit_transform(ml_df[col].astype(str))
    label_encoders[col] = le # We save the encoder to reverse the translation later if needed

# 4. Split data into Features (X) and Target (y)
X = ml_df[features]
y = ml_df[target]

# 5. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape} (Rows, Features)")
print(f"Testing data shape: {X_test.shape} (Rows, Features)")
print("\nLook at how the text was converted to numbers:")
display(X_train.head())

Training data shape: (144406, 5) (Rows, Features)
Testing data shape: (36102, 5) (Rows, Features)

Look at how the text was converted to numbers:


,shipping_mode,customer_segment,market,order_region,category_name
131559,3,1,1,17,34
109986,3,0,4,21,18
99898,3,0,4,18,34
115085,0,1,3,7,17
104460,3,0,1,22,9


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize the model
# n_estimators=100 means we are building 100 decision trees
# n_jobs=-1 tells your CPU to use all available cores to train faster
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training the Random Forest Model... (This might take 10-30 seconds)")

# 2. Train (fit) the model
rf_model.fit(X_train, y_train)

# 3. Make predictions on our test set
y_pred = rf_model.predict(X_test)

# 4. Evaluate the model (Phase 10 integration)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%\n")

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

Training the Random Forest Model... (This might take 10-30 seconds)

Model Accuracy: 68.76%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.61      0.83      0.71     16354
           1       0.80      0.57      0.66     19748

    accuracy                           0.69     36102
   macro avg       0.71      0.70      0.69     36102
weighted avg       0.72      0.69      0.68     36102



In [3]:
import joblib
import os

# 1. Ensure the models directory exists
os.makedirs('../models', exist_ok=True)

# 2. Save the trained Random Forest model
model_path = '../models/rf_delivery_model.joblib'
joblib.dump(rf_model, model_path)

# 3. Save the Label Encoders
encoders_path = '../models/label_encoders.joblib'
joblib.dump(label_encoders, encoders_path)

print(f"✅ Model successfully saved to: {model_path}")
print(f"✅ Encoders successfully saved to: {encoders_path}")

✅ Model successfully saved to: ../models/rf_delivery_model.joblib
✅ Encoders successfully saved to: ../models/label_encoders.joblib
